# CIESOL Anomaly Detection — Preprocessing (Step-by-Step)

This notebook implements the complete, modular preprocessing pipeline for the Agroconnect greenhouse dataset.

### Core Design Principles
1. **Real Timestamp Preservation**: Real Agroconnect inter-arrival time is ~30–31 s due to acquisition jitter. We do **not** force the series onto a global grid or resample.
2. **Selective Gap Reconstruction**: 30-second grid slots are inserted **only** inside genuine missing-record gaps (`gap_category` in `short_gap`, `medium_gap`, `long_gap`).
3. **Short-Gap Interpolation**: Only reconstructed timestamps inside short gaps (≤ 5.5 min) are linearly interpolated from the surrounding original boundary observations.
4. **Medium & Long Gaps Preserved**: Medium (> 5.5–10 min) and long (> 10 min) gaps remain `NaN` and are marked with `in_medium_gap`/`in_long_gap` and `sequence_unsafe = True`.
5. **Physical Value Purity**: `preprocessed_data.parquet` contains exclusively physical sensor measurements, timestamps, and quality/provenance flags (no scaled columns).
6. **Leakage-Safe Scaling**: Scaling and dataset splitting are decoupled and fitted strictly on the training set.

## 0. Setup & Configuration

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from src.preprocessing import (
    PreprocessConfig,
    SENSOR_COLUMNS,
    load_clean_data,
    analyze_duplicate_timestamps,
    analyze_time_gaps,
    create_gap_grid,
    interpolate_short_gaps,
    handle_medium_gaps,
    handle_long_gaps,
    update_sequence_safety,
    analyze_missing_values,
    check_physical_ranges,
    generate_quality_report,
    save_preprocessed,
    chronological_split,
    fit_scaler,
    transform_with_scaler,
)

# Initialize configuration
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "clean_data.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
QUALITY_DIR = PROJECT_ROOT / "data" / "quality"

cfg = PreprocessConfig(
    clean_path=CLEAN_PATH,
    output_dir=OUTPUT_DIR,
    quality_dir=QUALITY_DIR,
    expected_seconds=30.0,
    normal_interval_min=29.0,
    normal_interval_max=32.0,
    short_gap_minutes=5.5,
    medium_gap_minutes=10.0,
    train_days=60,
    validation_days=10,
    test_days=10,
)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Clean Data Path ({'exists' if CLEAN_PATH.exists() else 'missing'}): {CLEAN_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Quality Directory: {QUALITY_DIR}")

## 1. Load Clean Data

Loads `data/processed/clean_data.parquet` and validates column schema, datetime conversion, sorting, and monotonicity.

In [ ]:
df_clean = load_clean_data(cfg.clean_path)

print(f"Dataset Shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Date Range    : {df_clean['FECHA'].min()} → {df_clean['FECHA'].max()}")
print(f"Duration      : {(df_clean['FECHA'].max() - df_clean['FECHA'].min()).total_seconds() / 86400:.2f} days")
print(f"Missing NaNs  : {df_clean[SENSOR_COLUMNS].isna().sum().sum()}")
df_clean.head(5)

## 2. Duplicate Timestamps Analysis

- Identifies identical duplicate rows and safely deduplicates them.
- Identifies and reports conflicting duplicate timestamps (different sensor measurements on the same timestamp).

In [ ]:
dup_report = analyze_duplicate_timestamps(df_clean, remove_identical=True)
df = dup_report["df"]

print(f"Total Duplicate Rows        : {dup_report['n_duplicate_rows']}")
print(f"Duplicate Timestamps        : {dup_report['n_duplicate_timestamps']}")
print(f"Identical Groups (Removed)  : {dup_report['n_identical_groups']}")
print(f"Conflicting Groups (Report) : {dup_report['n_conflicting_groups']}")
if not dup_report["conflicts"].empty:
    display(dup_report["conflicts"])

## 3. Timestamp & Inter-Arrival Gap Analysis

Analyzes sampling intervals on the original observed timestamps:
- **29–32 s**: Normal sampling jitter band (>98% of observations)
- **>32 s to 5.5 min**: `short_gap`
- **>5.5 to 10 min**: `medium_gap`
- **>10 min**: `long_gap`

In [ ]:
time_analysis = analyze_time_gaps(df, cfg=cfg)
gap_report = time_analysis["gap_report"].copy()

print("=== Inter-Arrival Statistics ===")
print(f"Total Intervals               : {time_analysis['n_intervals']:,}")
print(f"Minimum Interval              : {time_analysis['min_interval']:.2f} s")
print(f"Median Interval               : {time_analysis['median_interval']:.2f} s")
print(f"Mean Interval                 : {time_analysis['mean_interval']:.2f} s")
print(f"P95 / P99 Interval            : {time_analysis['percentiles']['p95']:.2f} s / {time_analysis['percentiles']['p99']:.2f} s")
print(f"Maximum Interval              : {time_analysis['max_interval']:.2f} s")
print(f"Percentage in 30–31 s         : {time_analysis['pct_intervals_30_31']:.3f}%")
print(f"Percentage in Normal Band     : {time_analysis['pct_intervals_in_normal_band']:.3f}%")
print(f"Gap Candidates Count          : {time_analysis['n_gap_candidates']}")
print(f"Gap Category Breakdown        : {time_analysis['gap_category_counts']}")

In [ ]:
# Plot interval distribution around 30s nominal period
dist = time_analysis["interval_distribution"].copy()
plot_dist = dist[dist["interval_seconds"].between(25, 45)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(plot_dist["interval_seconds"], plot_dist["percentage"], width=0.8, color="#1f77b4", edgecolor="black", alpha=0.85)
ax.axvspan(cfg.normal_interval_min, cfg.normal_interval_max, color="green", alpha=0.18, label=f"Normal Band [{cfg.normal_interval_min}s, {cfg.normal_interval_max}s]")
ax.set_xlabel("Interval Duration (seconds)", fontsize=11)
ax.set_ylabel("Percentage of Observations (%)", fontsize=11)
ax.set_title("Agroconnect Sampling Interval Distribution (25–45 s Zoom)", fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("=== Detected Gap Episodes (Sample) ===")
display(gap_report.head(10))

In [ ]:
time_analysis = analyze_time_gaps(df, cfg=cfg)

gap_report = time_analysis["gap_report"].copy()

print(gap_report["gap_category"].value_counts())

## 4. Reconstruct Missing Timestamps Inside Gaps (`create_gap_grid`)

Inserts 30-second grid slots **strictly** inside genuine missing episodes (`short_gap`, `medium_gap`, `long_gap`).
- Reconstructed rows receive `timestamp_was_missing = True` and initial sensor values = `NaN`.
- Original observations are preserved unchanged with `timestamp_was_missing = False`.

In [ ]:
df_grid = create_gap_grid(
    df,
    gap_report,
    expected_seconds=cfg.expected_seconds,
    categories=["short_gap", "medium_gap", "long_gap"],
)

print(f"Rows Before Reconstruction  : {len(df):,}")
print(f"Rows After Reconstruction   : {len(df_grid):,}")
print(f"Original Observations       : {int(df_grid['_original_observation'].sum()):,}")
print(f"Reconstructed Timestamps    : {int(df_grid['timestamp_was_missing'].sum()):,}")

## 5. Short-Gap Linear Interpolation (`interpolate_short_gaps`)

- Interpolates **only** `short_gap` episodes (≤ 5.5 min).
- Interpolates **only** reconstructed timestamps (`timestamp_was_missing == True`).
- Uses the real observation immediately before and after each gap.
- Sets `<sensor>_was_missing = True` and `<sensor>_interpolated = True` for reconstructed slots.

In [ ]:
df_interp = interpolate_short_gaps(df_grid, gap_report=gap_report, cfg=cfg)

short_mask = df_interp["in_short_gap"]
print(f"Rows in Short Gaps          : {int(short_mask.sum()):,}")
print(f"PTEXT Interpolated Flag Count : {int(df_interp['PTEXT_interpolated'].sum()):,}")
print(f"PTEXT Missing Flag Count      : {int(df_interp['PTEXT_was_missing'].sum()):,}")

## 6. Medium & Long Gap Handling (`handle_medium_gaps`, `handle_long_gaps`)

- `handle_medium_gaps()` processes **only** `"medium_gap"`.
- `handle_long_gaps()` processes **only** `"long_gap"`.
- Missing sensor values inside medium and long gaps remain `NaN` (no interpolation).
- Flagged with `in_medium_gap`, `in_long_gap`, and `sequence_unsafe = True`.

In [ ]:
df_proc = handle_medium_gaps(df_interp, gap_report)
df_proc = handle_long_gaps(df_proc, gap_report)
df_proc = update_sequence_safety(df_proc)

print(f"In Medium Gap Rows : {int(df_proc['in_medium_gap'].sum()):,}")
print(f"In Long Gap Rows   : {int(df_proc['in_long_gap'].sum()):,}")
print(f"Sequence Unsafe    : {int(df_proc['sequence_unsafe'].sum()):,} ({100 * df_proc['sequence_unsafe'].mean():.3f}%)")

## 7. Verification of Known Gap Windows

Let's inspect the two prominent ~5 min gap episodes (October 17, 2020 and December 6, 2020) to verify reconstructed timestamps and interpolation.

In [ ]:
print("=== October 17, 2020 Gap Window ===")
oct_window = df_proc[
    (df_proc["FECHA"] >= "2020-10-17 04:37:00") &
    (df_proc["FECHA"] <= "2020-10-17 04:43:30")
]
display(oct_window[["FECHA", "timestamp_was_missing", "gap_category", "PTEXT", "PTEXT_interpolated", "sequence_unsafe"]])

print("\n=== December 6, 2020 Gap Window ===")
dec_window = df_proc[
    (df_proc["FECHA"] >= "2020-12-06 02:33:00") &
    (df_proc["FECHA"] <= "2020-12-06 02:40:00")
]
display(dec_window[["FECHA", "timestamp_was_missing", "gap_category", "PTEXT", "PTEXT_interpolated", "sequence_unsafe"]])

## 8. Missing-Value Analysis (Situation A vs B)

- **Situation A**: Sensor `NaN` on an original timestamp.
- **Situation B**: Reconstructed timestamp from missing records.

In [ ]:
missing_report = analyze_missing_values(df_proc)
display(missing_report)

## 9. Physical Range Checks

Flags out-of-range physical readings based on sensor instrument datasheets (`COLUMN_PHYSICAL_RANGES`). Does **not** modify or delete values.

In [ ]:
df_proc, physical_summary = check_physical_ranges(df_proc)
print(f"Total Out-of-Range Flags: {int(physical_summary['out_of_range'].sum()):,}")
display(physical_summary)

## 10. Quality Report Generation & Persistence

Generates the standardized tabular data quality report and persists `preprocessed_data.parquet` containing purely physical values and flags (no scaled columns).

In [ ]:
quality_report = generate_quality_report(
    df=df_proc,
    time_analysis=time_analysis,
    gap_report=gap_report,
    missing_report=missing_report,
    duplicate_report=dup_report,
    physical_summary=physical_summary,
)

saved_paths = save_preprocessed(
    df=df_proc,
    gap_report=gap_report,
    quality_report=quality_report,
    time_analysis=time_analysis,
    output_dir=cfg.output_dir,
    quality_dir=cfg.quality_dir,
)

print("=== Persisted Preprocessing Artefacts ===")
for key, path in saved_paths.items():
    print(f"  {key:22s}: {path}")

# Verify preprocessed parquet content
df_verify = pd.read_parquet(saved_paths["preprocessed"])
scaled_cols = [c for c in df_verify.columns if c.endswith("_scaled")]
print(f"\nVerification of preprocessed_data.parquet:")
print(f"  Total Rows    : {len(df_verify):,}")
print(f"  Total Columns : {len(df_verify.columns)}")
print(f"  Scaled Columns: {scaled_cols} (Should be empty)")
assert len(scaled_cols) == 0, "preprocessed_data.parquet must not contain scaled columns!"

In [ ]:
short_gaps = gap_report[
    gap_report["gap_category"] == "short_gap"
]

print("Number of short gaps:", len(short_gaps))
print(
    short_gaps[
        ["gap_start", "gap_end",
         "duration_seconds",
         "estimated_missing_samples"]
    ].to_string(index=False)
)

print(
    "Total estimated missing samples:",
    short_gaps["estimated_missing_samples"].sum()
)

In [ ]:
print("=== FINAL GAP VALIDATION ===")

print("Timing jitter intervals:",
      (gap_report["gap_category"] == "timing_jitter").sum())

print("Short gaps:",
      (gap_report["gap_category"] == "short_gap").sum())

print("Medium gaps:",
      (gap_report["gap_category"] == "medium_gap").sum())

print("Long gaps:",
      (gap_report["gap_category"] == "long_gap").sum())

print("Reconstructed timestamps:",
      int(df_proc["timestamp_was_missing"].sum()))

print("Total rows:",
      len(df_proc))

## 11. Decoupled Splitting & Scaling Preview (`05_data_splitting.ipynb`)

Demonstrating how `chronological_split()`, `fit_scaler()`, and `transform_with_scaler()` are applied downstream in notebook `05_data_splitting.ipynb`.
- **Fit ONLY on Train**
- **Transform Train, Validation, and Test**

In [ ]:
# 1. Chronological Split (60d train / 10d validation / 10d test)
splits = chronological_split(
    df_proc,
    train_days=cfg.train_days,
    validation_days=cfg.validation_days,
    test_days=cfg.test_days,
)

# Unscaled splits (kept in memory only)
train_unscaled = splits["train"]
val_unscaled = splits["validation"]
test_unscaled = splits["test"]

# 2. Fit scaler strictly on complete train rows
scaler = fit_scaler(train_unscaled)

# 3. Transform splits using the train-fitted scaler
train_scaled = transform_with_scaler(train_unscaled, scaler)
val_scaled = transform_with_scaler(val_unscaled, scaler)
test_scaled = transform_with_scaler(test_unscaled, scaler)

print("=== Downstream Splits Summary ===")
for name, part in [("Train", train_scaled), ("Validation", val_scaled), ("Test", test_scaled)]:
    print(f"  {name:12s}: {part['FECHA'].min()} → {part['FECHA'].max()} ({len(part):,} rows)")
    print(f"    Mean XTINV_scaled = {part['XTINV_scaled'].mean():.4f}, Std XTINV_scaled = {part['XTINV_scaled'].std():.4f}")

In [ ]:
print("=== SPLIT OVERLAP CHECK ===")

train_end = train_scaled["FECHA"].max()
val_start = val_scaled["FECHA"].min()
val_end = val_scaled["FECHA"].max()
test_start = test_scaled["FECHA"].min()

print("Train end      :", train_end)
print("Validation start:", val_start)
print("Validation end :", val_end)
print("Test start     :", test_start)

print("\nOverlap checks:")
print("Train/Val overlap:", train_end >= val_start)
print("Val/Test overlap :", val_end >= test_start)

In [ ]:
print("\n=== ROW COUNTS ===")
print("Train      :", len(train_scaled))
print("Validation :", len(val_scaled))
print("Test       :", len(test_scaled))
print("Total      :", len(train_scaled) + len(val_scaled) + len(test_scaled))
print("Original   :", len(df_proc))

In [ ]:
from pathlib import Path
import joblib

# Output directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the three unscaled splits
train_unscaled_path = processed_dir / "train_unscaled.parquet"
val_unscaled_path = processed_dir / "validation_unscaled.parquet"
test_unscaled_path = processed_dir / "test_unscaled.parquet"

train_unscaled.to_parquet(train_unscaled_path, index=False)
val_unscaled.to_parquet(val_unscaled_path, index=False)
test_unscaled.to_parquet(test_unscaled_path, index=False)

# Save the three scaled splits
train_path = processed_dir / "train_scaled.parquet"
val_path = processed_dir / "validation_scaled.parquet"
test_path = processed_dir / "test_scaled.parquet"

train_scaled.to_parquet(train_path, index=False)
val_scaled.to_parquet(val_path, index=False)
test_scaled.to_parquet(test_path, index=False)

# Save the fitted scaler
scaler_path = processed_dir / "scaler.pkl"
joblib.dump(scaler, scaler_path)

print("=== Saved Files ===")
print(f"Train (unscaled)      : {train_unscaled_path}")
print(f"Validation (unscaled) : {val_unscaled_path}")
print(f"Test (unscaled)       : {test_unscaled_path}")
print(f"Train (scaled)        : {train_path}")
print(f"Validation (scaled)   : {val_path}")
print(f"Test (scaled)         : {test_path}")
print(f"Scaler                : {scaler_path}")